In [1]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 67.5 MB/s eta 0:00:00


In [2]:
import mysql.connector
from mysql.connector import Error

hostname = "nxhwel.h.filess.io"
database = "olist_project_correctly"
port = "3306"
username = "olist_project_correctly"
password = "6466fc1173b43d3a21028a41a65402267046a393"

try:
    connection = mysql.connector.connect(host=hostname, database=database, user=username, password=password, port=port)
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        cursor = connection.cursor()
        cursor.execute("select database();")
        record = cursor.fetchone()
        print("You're connected to database: ", record)

except Error as e:
    print("Error while connecting to MySQL", e)
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL connection is closed")



/tmp/ipykernel_8356/3432079683.py:13: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()


Connected to MySQL Server version  5.7.38-41
You're connected to database:  ('olist_project_correctly',)
MySQL connection is closed


In [4]:
import pandas as pd
order_payments = pd.read_csv('/content/sample_data/olist_order_payments_dataset.csv')

In [6]:
order_payments.shape

(103886, 5)

In [8]:
import mysql.connector
from mysql.connector import Error

# Connection details
hostname = "nxhwel.h.filess.io"
database = "olist_project_correctly"
port = "3306"
username = "olist_project_correctly"
password = "6466fc1173b43d3a21028a41a65402267046a393"

# CSV file path
csv_file_path = "/content/sample_data/olist_order_payments_dataset.csv"

# Table name where the data will be uploaded
table_name = "olist_order_payments"

try:
    # Step 1: Establish a connection to MySQL server
    connection = mysql.connector.connect(
        host=hostname,
        database=database,
        user=username,
        password=password,
        port=port
    )
    if connection.is_connected():
        print("Connected to MySQL Server successfully!")

        # Step 2: Create a cursor to execute SQL queries
        cursor = connection.cursor()

        # Step 3: Drop table if it already exists (for clean insertion)
        cursor.execute(f"DROP TABLE IF EXISTS {table_name};")
        print(f"Table `{table_name}` dropped if it existed.")

        # Step 4: Create a table structure to match CSV file
        create_table_query = f"""
        CREATE TABLE {table_name} (
            order_id VARCHAR(50),
            payment_sequential INT,
            payment_type VARCHAR(20),
            payment_installments INT,
            payment_value FLOAT
        );
        """
        cursor.execute(create_table_query)
        print(f"Table `{table_name}` created successfully!")

        # Step 5: Load the CSV data into pandas DataFrame
        data = pd.read_csv(csv_file_path)
        print("CSV data loaded into pandas DataFrame.")

        # Step 6: Insert data in batches of 500 records
        batch_size = 10000  # Define the batch size
        total_records = len(data)  # Get total records in the DataFrame

        print(f"Starting data insertion into `{table_name}` in batches of {batch_size} records.")
        for start in range(0, total_records, batch_size):
            end = start + batch_size
            batch = data.iloc[start:end]  # Get the current batch of records

            # Convert batch to list of tuples for MySQL insertion
            batch_records = [
                tuple(row) for row in batch.itertuples(index=False, name=None)
            ]

            # Prepare the INSERT query
            insert_query = f"""
            INSERT INTO {table_name}
            (order_id, payment_sequential, payment_type, payment_installments, payment_value)
            VALUES (%s, %s, %s, %s, %s);
            """

            # Execute the insertion query for the batch
            cursor.executemany(insert_query, batch_records)
            connection.commit()  # Commit after each batch
            print(f"Inserted records {start + 1} to {min(end, total_records)} successfully.")

        print(f"All {total_records} records inserted successfully into `{table_name}`.")

except Error as e:
    # Step 7: Handle any errors
    print("Error while connecting to MySQL or inserting data:", e)

finally:
    # Step 8: Close the cursor and connection
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL connection is closed.")

Connected to MySQL Server successfully!
Table `olist_order_payments` dropped if it existed.
Table `olist_order_payments` created successfully!
CSV data loaded into pandas DataFrame.
Starting data insertion into `olist_order_payments` in batches of 10000 records.
Inserted records 1 to 10000 successfully.
Inserted records 10001 to 20000 successfully.
Inserted records 20001 to 30000 successfully.
Inserted records 30001 to 40000 successfully.
Inserted records 40001 to 50000 successfully.
Inserted records 50001 to 60000 successfully.
Inserted records 60001 to 70000 successfully.
Inserted records 70001 to 80000 successfully.
Inserted records 80001 to 90000 successfully.
Inserted records 90001 to 100000 successfully.
Inserted records 100001 to 103886 successfully.
All 103886 records inserted successfully into `olist_order_payments`.
MySQL connection is closed.
